# Behaviour-shaped encoder: resultsRegenerates every encoder/decoder table and figure in `paper/paper.md` §6 from committed artifacts.Nothing here is typed by hand: if a number disagrees with the paper, the paper is stale.**Inputs** — `checkpoints/bc/bc-decode-final.pt` (metrics + weights) and `results/event_detection.json`.Regenerate those with:```sh.venv/bin/python scripts/train_bc_decode.py --epochs 14 --oversample 0.05 --w-consistency 0.5 \    --out checkpoints/bc/bc-decode-final.pt.venv/bin/python scripts/eval_decode_events.py checkpoints/bc/bc-decode-final.pt```Split is **by start slot, three ways**: train {1,2,3,5,6,10}, threshold-tuning {4,8}, test {7,9}.Everything reported is test-slot; operating points come from the tuning slots only.

In [ ]:
import json, sysfrom pathlib import Pathimport numpy as np, torchimport matplotlib.pyplot as pltROOT = Path.cwd().parent if Path.cwd().name == "results" else Path.cwd()sys.path.insert(0, str(ROOT))FIG = ROOT / "results/figures"; FIG.mkdir(parents=True, exist_ok=True)plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": .3,                     "axes.spines.top": False, "axes.spines.right": False})COLORS = {"model": "#2171b5", "baseline": "#bdbdbd", "good": "#238b45", "bad": "#cb181d"}CK = torch.load(ROOT / "checkpoints/bc/bc-decode-final.pt", weights_only=False, map_location="cpu")EV = json.loads((ROOT / "results/event_detection.json").read_text())print("split:", CK["split"])print("latent dim:", CK["latent"], "| action top-1:", round(CK["metrics"]["top1"], 4))

## Table 4 — what survives the poolingThe concern was that masked mean+max pooling over actor rows destroys per-row identity, making per-enemyHP and behaviour state unrecoverable from `z`. Accuracy columns are against a majority-class baseline;MAE columns against predict-the-train-mean.

In [ ]:
rows = CK["metrics"]["fields"]print(f"{'field':<28} {'metric':<16} {'model':>9} {'baseline':>9}  unit")print("-" * 78)for r in rows:    print(f"{r['field']:<28} {r['metric']:<16} {r['model']:>9.4f} {r['baseline']:>9.4f}  {r['unit']}")

In [ ]:
# accuracy fields: model vs baseline, sorted by headroom capturedacc = [r for r in rows if r["metric"] == "accuracy" and r["baseline"] < 0.999]acc.sort(key=lambda r: r["baseline"])y = np.arange(len(acc))fig, ax = plt.subplots(figsize=(7, 0.42 * len(acc) + 1.2))ax.barh(y + .18, [r["model"] for r in acc], .36, label="model", color=COLORS["model"])ax.barh(y - .18, [r["baseline"] for r in acc], .36, label="majority baseline", color=COLORS["baseline"])ax.set_yticks(y, [r["field"] for r in acc]); ax.set_xlim(0, 1.02)ax.set_xlabel("accuracy (test slots)"); ax.set_title("Reward-function inputs decoded from a frozen 128-d latent")ax.legend(loc="lower right", fontsize=8)fig.tight_layout(); fig.savefig(FIG / "decode_fields.png", bbox_inches="tight", dpi=200)print("->", FIG / "decode_fields.png")

In [ ]:
# positional fields, against the scale of the motion they have to resolvepos = [r for r in rows if "pos" in r["field"]]mot = {"Link per-step p90": 9.75, "Link 50-step p90": 165.5}   # measured on the datasetfig, ax = plt.subplots(figsize=(7, 3.2))lab = [r["field"] for r in pos]; val = [r["model"] for r in pos]ax.barh(np.arange(len(pos)), val, .6, color=COLORS["model"])for k, (n, v) in enumerate(mot.items()):    ax.axvline(v, ls="--", lw=1.2, color=["#cb181d", "#756bb1"][k], label=f"{n} = {v}")ax.set_yticks(np.arange(len(pos)), lab); ax.set_xscale("log")ax.set_xlabel("error, world units (log)"); ax.legend(fontsize=7)ax.set_title("Positional resolution vs the motion it must resolve")fig.tight_layout(); fig.savefig(FIG / "decode_position.png", bbox_inches="tight", dpi=200)print({r["field"]: round(r["model"], 2) for r in pos})

## Table 5 — reward-critical event detectionThe reward reads HP **deltas**, and damage happens once in 253 transitions. At a per-frame HP error rate`eps`, each error fabricates ~2 spurious deltas, so precision ~ `0.0019 / (0.0019 + 2*eps)` — three to fournines of per-frame accuracy are needed before differencing a decoded state gives usable events.**AP is threshold-free and is the number to compare.** F1/precision/recall are at a threshold fitted on thetuning slots, which for Link damage did not transfer.

In [ ]:
print(f"{'event':<13} {'detector':<26} {'F1':>6} {'prec':>6} {'rec':>6} {'AP':>6}  {'TP/FP/FN':>11}")print("-" * 82)for ev, dets in EV["events"].items():    for name, r in dets.items():        print(f"{ev:<13} {name:<26} {r['f1']:>6.3f} {r['precision']:>6.3f} {r['recall']:>6.3f} "              f"{r['ap']:>6.3f}  {r['tp']:>3}/{r['fp']:>3}/{r['fn']:<3}")    print()

In [ ]:
# best detector per event, by APbest = {ev: max(d.items(), key=lambda kv: kv[1]["ap"]) for ev, d in EV["events"].items()}fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.4))names = list(best); x = np.arange(len(names))a1.bar(x - .2, [best[n][1]["precision"] for n in names], .38, label="precision", color=COLORS["model"])a1.bar(x + .2, [best[n][1]["recall"] for n in names], .38, label="recall", color=COLORS["good"])a1.set_xticks(x, names); a1.set_ylim(0, 1.05); a1.legend(fontsize=8)a1.set_title("Best detector per event (threshold from tuning slots)")a2.bar(x, [best[n][1]["ap"] for n in names], .55, color=COLORS["model"])for i, n in enumerate(names):    a2.text(i, best[n][1]["ap"] + .02, f"{best[n][1]['ap']:.3f}", ha="center", fontsize=8)    a2.text(i, .04, f"n={best[n][1]['true_events']}", ha="center", fontsize=7, color="white")a2.set_xticks(x, names); a2.set_ylim(0, 1.05); a2.set_ylabel("average precision")a2.set_title("Threshold-free: AP (test events are few)")fig.tight_layout(); fig.savefig(FIG / "event_detection.png", bbox_inches="tight", dpi=200)print({n: (best[n][0], round(best[n][1]["ap"], 3)) for n in names})

## Caveats that belong next to these numbers- **Upper bound.** These autoencode `s_t` through `z_t = E(s_t)`. A dynamics model decodes from a  *predicted* `z_{t+1}` and can only do worse.- **Small test counts** — 28 damage, 19 kill, 24 Link-damage events. Differences of a few points are noise.- **Thresholds transfer imperfectly.** Link damage: recall 0.92 on the tuning slots, 0.46 on test, precision  1.000 throughout. Tuning on the test split had inflated its F1 from 0.629 to 0.898.- **Reconstruction competes with action prediction.** Action top-1 is 0.049 against a 0.220 Bayes ceiling  (the demonstrator is stochastic: entropy 3.18 nats, ~24 effective actions) and a 0.039 majority baseline.